# Steward Export — Lightweight Artifacts (Flow + Transport)

**Role:** the group **steward** runs this notebook, once, **after** the master flow
(`case_study_flow_group_0.ipynb`) and master transport
(`case_study_transport_group_0.ipynb`) notebooks have been run to completion in
**this** group folder (so the heavy model workspaces exist on disk).

**What it does:** it reads the heavy model workspaces and writes a small,
**portable** `exports/` bundle (GeoPackages + CSV + JSON) into this folder. That
bundle — and *not* the heavy workspaces — travels inside the submission ZIP and is
the only thing the FloPy-free **scratch** notebook (`scratch_analysis_template.ipynb`)
needs to rerun.

**This notebook may import FloPy** (it lives in the group folder, which can reach the
course helpers at `../../../_SUPPORT/src`). The scratch notebook must not.

> Master and steward notebooks are **saved-output / provenance records**. The
> **scratch** notebook is the TA rerun target. See `COLLABORATION.md`.

In [ ]:
# --- Imports (FloPy is allowed here; the scratch notebook stays FloPy-free) ---
import os, sys, json, shutil, platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import yaml

# Repo helpers live three levels up from PROJECT/workspace/group_x/
sys.path.append(os.path.abspath('../../../_SUPPORT/src'))
import flopy

print('flopy', flopy.__version__, '| geopandas', gpd.__version__, '| numpy', np.__version__)

## §1 — Resolve group number and heavy workspaces

Reads `case_config.yaml` (+ `case_config_transport.yaml`) and reconstructs the same
output workspaces the master notebooks wrote to. Nothing here re-runs a model.

In [ ]:
# Read configs from THIS group folder
with open('case_config.yaml') as f:
    cfg = yaml.safe_load(f)

group_number = cfg['group'].get('number', 0)
if not isinstance(group_number, int) or not (0 <= group_number <= 8):
    raise ValueError(f'Group number must be an integer 0-8, got {group_number!r}')

namefile   = cfg['model']['namefile']            # e.g. limmat_valley_model_nwt.nam
model_name = namefile.replace('.nam', '')

# Flow workspaces (mirror the master flow notebook's layout)
parent_base_ws  = os.path.expanduser(cfg['output']['workspace']) + str(group_number) + '/parent_base'
def _ws(tag):
    return parent_base_ws.replace('parent_base', tag)
flow_ws = {
    'parent_base':     parent_base_ws,
    'parent_scenario': _ws('parent_scenario'),
    'sub_base':        _ws('sub_base'),
    'sub_wells':       _ws('sub_wells'),
    'sub_scenario':    _ws('sub_scenario'),
}
grids_ws = _ws('grids')
grid_gpkg = os.path.join(grids_ws, 'sub_model_grid.gpkg')

# Transport workspace (mirror the master transport notebook)
try:
    with open('case_config_transport.yaml') as f:
        cfg_t = yaml.safe_load(f)
    transport_ws = os.path.expanduser(cfg_t['output']['workspace'] + str(group_number))
except FileNotFoundError:
    cfg_t, transport_ws = None, None

# Local, portable output bundle (this is what goes into the submission ZIP)
EXPORTS = Path('exports')
EXPORTS.mkdir(exist_ok=True)

print(f'Group {group_number} | model {model_name}')
print('Grid GPKG :', grid_gpkg, '(exists)' if os.path.exists(grid_gpkg) else '(MISSING)')
print('Transport :', transport_ws)
print('Exports -> ', EXPORTS.resolve())

manifest = {}   # filename -> {present, ...}
missing_optional = []

## §2 — Flow heads → GeoPackages

Geometry comes from the submodel **grid GeoPackage** the master flow notebook
exported (polygon cells, `row`/`col`, EPSG:2056). Heads are read from each model's
`.hds`, masked with the model's **actual** `hdry` (`m.upw.hdry`) and `hnoflo`
(`m.bas6.hnoflo`) values, and joined onto the grid by `(row, col)`. The submodel is
single-layer, so we use `head[0]`. Using the exported grid geometry keeps world
coordinates correct without depending on a reloaded model's georeference.

In [ ]:
# Load the submodel grid geometry once (shared by base / wells / scenario)
if not os.path.exists(grid_gpkg):
    raise FileNotFoundError(
        f'Required grid GeoPackage not found: {grid_gpkg}\n'
        'Run the master FLOW notebook to completion first (it exports sub_model_grid.gpkg).')

grid_gdf = gpd.read_file(grid_gpkg)
grid_gdf = grid_gdf.rename(columns={'cell_id': 'cellid'})
if grid_gdf.crs is None or grid_gdf.crs.to_epsg() != 2056:
    grid_gdf = grid_gdf.set_crs(2056, allow_override=True)
nrow = int(grid_gdf['row'].max()) + 1
ncol = int(grid_gdf['col'].max()) + 1
print(f'Grid cells: {len(grid_gdf)}  ({nrow} rows x {ncol} cols)  CRS EPSG:{grid_gdf.crs.to_epsg()}')

head_mask_values = {}   # filled from the first model we load

def _load_masking(ws):
    """Load a submodel just to read its actual hdry / hnoflo values."""
    m = flopy.modflow.Modflow.load(namefile, model_ws=ws, forgive=True, check=False,
                                   load_only=['UPW', 'BAS6', 'DIS'], exe_name='mfnwt')
    hdry  = float(np.array(m.upw.hdry).flatten()[0])
    hnoflo = float(np.array(m.bas6.hnoflo).flatten()[0])
    return hdry, hnoflo

def export_heads(tag, out_name, required):
    ws = flow_ws[tag]
    hds_path = os.path.join(ws, f'{model_name}.hds')
    if not os.path.exists(hds_path):
        if required:
            raise FileNotFoundError(
                f'Required heads file missing: {hds_path}. Re-run the master flow notebook.')
        print(f'  [skip] {out_name}: no heads at {hds_path}')
        missing_optional.append(out_name)
        return None
    hdry, hnoflo = _load_masking(ws)
    head_mask_values.setdefault('hdry', hdry)
    head_mask_values.setdefault('hnoflo', hnoflo)

    head = flopy.utils.HeadFile(hds_path).get_data()[0]   # single layer -> head[0]
    g = grid_gdf.copy()
    vals = head[g['row'].to_numpy(), g['col'].to_numpy()].astype(float)
    dry = np.isclose(vals, hdry) | np.isclose(vals, hnoflo)
    vals[dry] = np.nan
    g['head_m'] = vals
    g = g[['cellid', 'row', 'col', 'head_m', 'geometry']]

    out_path = EXPORTS / out_name
    if out_path.exists():
        out_path.unlink()
    g.to_file(out_path, driver='GPKG')
    manifest[out_name] = {'present': True, 'rows': int(len(g)),
                          'n_active': int(np.isfinite(vals).sum())}
    print(f'  [ok] {out_name}: {len(g)} cells, {int(np.isfinite(vals).sum())} active '
          f'(hdry={hdry:g}, hnoflo={hnoflo:g})')
    return out_path

export_heads('sub_base',     'flow_heads_sub_base.gpkg',     required=True)
export_heads('sub_wells',    'flow_heads_sub_wells.gpkg',    required=True)
export_heads('sub_scenario', 'flow_heads_sub_scenario.gpkg', required=False)

## §3 — Flow budget → CSV

Reads each model's list file (derived robustly as `<model>.list`, **not** `.lst`) and
writes a tidy `flow_budget_summary.csv`. Parent models are included as well as the
submodels, so the river-leakage term is available for the budget/river-exchange card
(the CHD-bounded submodels do not carry a river term).

In [ ]:
def _tidy_budget(model_tag, ws):
    list_path = os.path.join(ws, f'{model_name}.list')   # robust: '.list', never '.lst'
    if not os.path.exists(list_path):
        return []
    try:
        # Mirror plot_utils.visualize_budget: get_budget()[-1] is a recarray with
        # named term fields (e.g. WELLS_IN, RIVER_LEAKAGE_OUT), one row per step.
        budget = flopy.utils.MfListBudget(list_path).get_budget()[-1]
        series = pd.DataFrame(budget).T.iloc[:, -1]   # index=term names -> last step values
    except Exception as exc:
        print(f'  [warn] could not read budget for {model_tag}: {exc}')
        return []
    rows = {}
    for name, value in series.items():
        name = str(name)
        if name.endswith('_IN'):
            rows.setdefault(name[:-3], {})['flow_in_m3_d'] = float(value)
        elif name.endswith('_OUT'):
            rows.setdefault(name[:-4], {})['flow_out_m3_d'] = float(value)
    out = []
    for term, d in rows.items():
        fin = d.get('flow_in_m3_d', 0.0); fout = d.get('flow_out_m3_d', 0.0)
        out.append({'model': model_tag, 'term': term,
                    'flow_in_m3_d': fin, 'flow_out_m3_d': fout,
                    'net_m3_d': fin - fout})
    return out

budget_rows = []
for tag, ws in flow_ws.items():
    budget_rows += _tidy_budget(tag, ws)

if not budget_rows:
    raise RuntimeError('No budget terms found in any flow workspace — re-run the master flow notebook.')

budget_df = pd.DataFrame(budget_rows)
budget_path = EXPORTS / 'flow_budget_summary.csv'
budget_df.to_csv(budget_path, index=False)
manifest['flow_budget_summary.csv'] = {'present': True, 'rows': int(len(budget_df)),
                                        'models': sorted(budget_df['model'].unique())}
print(f'  [ok] flow_budget_summary.csv: {len(budget_df)} rows, models '
      f"{sorted(budget_df['model'].unique())}")
budget_df.head(12)

## §4 — Transport breakthrough → CSV + metadata JSON

Reads `base_cache.npz` **directly with numpy** (not through `load_doublet_base`, which
would import the FloPy simulation) and writes the breakthrough curve and a small
metadata record. Transport is optional — if the group did not run it, this section
skips cleanly.

In [ ]:
def export_transport():
    if transport_ws is None:
        print('  [skip] transport: no case_config_transport.yaml')
        return
    npz_path = os.path.join(transport_ws, 'base_cache.npz')
    if not os.path.exists(npz_path):
        print(f'  [skip] transport: no base_cache.npz at {npz_path}')
        missing_optional.append('transport_breakthrough.csv')
        return
    z = np.load(npz_path, allow_pickle=True)
    times = np.asarray(z['times'], dtype=float)
    bt    = np.asarray(z['bt'], dtype=float)

    pd.DataFrame({'time_days': times, 'concentration_mg_L': bt}).to_csv(
        EXPORTS / 'transport_breakthrough.csv', index=False)
    manifest['transport_breakthrough.csv'] = {'present': True, 'rows': int(len(times))}

    # Scenario metadata + a faithful copy of the notebook's verdict inputs
    scn = None
    if cfg_t is not None:
        scn = {o['id']: o for o in cfg_t['transport_scenarios']['options']}.get(group_number)
    threshold = float(scn['monitoring']['threshold_mg_L']) if scn else None
    c_src     = float(scn['source']['concentration_mg_L']) if scn else None
    peak   = float(np.nanmax(bt)) if bt.size else None
    t_peak = float(times[int(np.nanargmax(bt))]) if bt.size else None
    t_exceed = None
    if threshold is not None and bt.size:
        above = np.where(bt >= threshold)[0]
        t_exceed = float(times[above[0]]) if len(above) else None

    meta = {
        'schema_version': '1.0',
        'group_number': group_number,
        'concession': (scn or {}).get('concession'),
        'contaminant': (scn or {}).get('contaminant'),
        'title': (scn or {}).get('title'),
        'threshold_mg_L': threshold,
        'c_src_mg_L': c_src,
        'peak_mg_L': peak,
        't_peak_days': t_peak,
        't_exceedance_days': t_exceed,
        'recirc_fraction': float(z['recirc']) if 'recirc' in z else None,
        'src_xy': [float(x) for x in np.asarray(z['src_xy']).ravel()] if 'src_xy' in z else None,
        'receptor_xy': [float(x) for x in np.asarray(z['receptor_xy']).ravel()] if 'receptor_xy' in z else None,
        'base_cache_meta': dict(z['meta'].item()) if 'meta' in z else {},
    }
    with open(EXPORTS / 'transport_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    manifest['transport_meta.json'] = {'present': True}
    print(f'  [ok] transport_breakthrough.csv ({len(times)} steps) + transport_meta.json '
          f'(peak {peak:.4g} mg/L, threshold {threshold})')

export_transport()

## §5 — Optional: MODPATH pathlines → CSV

Best-effort only. If the master flow notebook ran the optional MODPATH particle
tracking (Section 3.5), summarise it; otherwise skip cleanly. Guarded so a missing or
unreadable pathline file never fails the export.

In [ ]:
def export_pathlines():
    ws = flow_ws['sub_wells']
    fpth = os.path.join(ws, f'{model_name}_mp.mppth')
    if not os.path.exists(fpth):
        print('  [skip] pathlines: no MODPATH .mppth (optional Section 3.5 not run)')
        missing_optional.append('pathlines_summary.csv')
        return
    try:
        pl = flopy.utils.PathlineFile(fpth).get_alldata()
        n_particles = len(pl)
        # travel time = last time along each particle path
        end_times = np.array([float(p['time'][-1]) for p in pl if len(p)]) if n_particles else np.array([])
        rows = [
            {'metric': 'n_particles',        'value': float(n_particles)},
            {'metric': 'max_travel_time_d',  'value': float(end_times.max()) if end_times.size else np.nan},
            {'metric': 'mean_travel_time_d', 'value': float(end_times.mean()) if end_times.size else np.nan},
            {'metric': 'min_travel_time_d',  'value': float(end_times.min()) if end_times.size else np.nan},
        ]
        pd.DataFrame(rows).to_csv(EXPORTS / 'pathlines_summary.csv', index=False)
        manifest['pathlines_summary.csv'] = {'present': True, 'n_particles': int(n_particles)}
        print(f'  [ok] pathlines_summary.csv: {n_particles} particles')
    except Exception as exc:
        print(f'  [skip] pathlines: could not summarise ({exc})')
        missing_optional.append('pathlines_summary.csv')

export_pathlines()

## §6 — Provenance manifest → `run_info.json`

Writes the schema version, group number, scenario metadata, timestamp, package
versions, head-masking values, CRS, grid shape and the export manifest. The scratch
notebook validates this with `scratch_io.check_schema()`.

In [ ]:
def _ver(mod):
    return getattr(mod, '__version__', 'unknown')

flow_scn = None
try:
    flow_scn = {o['id']: o for o in cfg['scenarios']['options']}.get(group_number)
except Exception:
    pass

run_info = {
    'schema_version': '1.0',
    'group_number': group_number,
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'crs': 'EPSG:2056',
    'grid': {'nrow': nrow, 'ncol': ncol, 'ncell': int(len(grid_gdf))},
    'head_masking': head_mask_values,
    'model_name': model_name,
    'scenario': {
        'flow': {'concession': (flow_scn or {}).get('concession'),
                 'label': (flow_scn or {}).get('label'),
                 'type': (flow_scn or {}).get('type')},
        'transport': ({'concession': cfg_t and None} if cfg_t is None else
                      {k: ({o['id']: o for o in cfg_t['transport_scenarios']['options']}
                           .get(group_number, {}) or {}).get(k)
                       for k in ('concession', 'contaminant', 'title')}),
    },
    'package_versions': {
        'python': platform.python_version(),
        'numpy': _ver(np), 'pandas': _ver(pd), 'geopandas': _ver(gpd),
        'flopy': _ver(flopy),
    },
    'exports': manifest,
    'missing_optional': sorted(set(missing_optional)),
}

with open(EXPORTS / 'run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2, default=str)

print('Wrote', (EXPORTS / 'run_info.json').resolve())
print(json.dumps(run_info, indent=2, default=str))

## Done

The `exports/` folder now holds the lightweight bundle. Next steps:

1. Commit / freeze the exports at least **3 days before submission** (see
   `COLLABORATION.md`).
2. Each card owner opens `scratch_analysis_template.ipynb`, sets their `CARD`, and
   produces figures/tables from this bundle — no FloPy, no heavy workspace.
3. Zip the group folder (**exclude** the heavy model workspaces under
   `~/applied_groundwater_modelling_data/`) and submit via Moodle. See
   `SUBMISSION_README_TEMPLATE.md`.